In [ ]:
from datetime import datetime, timedelta

In [ ]:
# 날짜 계산 로직
today = datetime.today()
monday_of_week = today - timedelta(days=today.weekday())
base_date = monday_of_week - timedelta(weeks=45)

if today.weekday() == 1:  # 화요일
    start_date = base_date
    end_date = base_date + timedelta(days=2)
    
elif today.weekday() == 3:  # 목요일
    start_date = base_date + timedelta(days=3)
    end_date = base_date + timedelta(days=6)
    
else:
    start_date = end_date = None

print(f"조회시작일자: {start_date.strftime('%Y-%m-%d') if start_date else None}")
print(f"조회종료일자: {end_date.strftime('%Y-%m-%d') if end_date else None}")

In [ ]:
# 2025년 누락일 검증
def get_dates_for_day(today):
    monday = today - timedelta(days=today.weekday())
    base_date = monday - timedelta(weeks=45)

    if today.weekday() == 1:  # 화요일
        return [base_date + timedelta(days=i) for i in range(3)]  # 월~수
    elif today.weekday() == 3:  # 목요일
        return [base_date + timedelta(days=i) for i in range(3, 7)]  # 목~일
    return []

start = datetime(2025, 1, 1)
end = datetime(2025, 12, 31)

collected_days = []

current = start
while current <= end:
    dates = get_dates_for_day(current)
    collected_days.extend(dates)
    current += timedelta(days=1)

# 중복 제거 및 정렬
collected_days = sorted(set(collected_days))

missing = []
for i in range(len(collected_days) - 1):
    diff = (collected_days[i+1] - collected_days[i]).days
    if diff != 1:
        missing.append((collected_days[i], collected_days[i+1], diff))

if not missing:
    print("✅ 모든 날짜가 연속적으로 커버되었습니다. (누락 없음)")
else:
    print("❌ 누락 발생!")
    for m in missing:
        print(f"누락 구간: {m[0]} → {m[1]} (차이: {m[2]}일)")

In [ ]:
# 누락일 발생 검증(4년)
def check_coverage_v2(start_year, end_year):
    all_dates = set()
    date_sources = {}  # 어느 발송일에서 조회됐는지 추적
    overlaps = []
    
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            for day in range(1, 32):
                try:
                    today = datetime(year, month, day)
                    
                    if today.weekday() in [1, 3]:  # 화요일 또는 목요일
                        monday_of_week = today - timedelta(days=today.weekday())
                        base_date = monday_of_week - timedelta(weeks=45)
                        
                        if today.weekday() == 1:  # 화요일
                            dates = [base_date + timedelta(days=i) for i in range(4)]  # 월~목
                        else:  # 목요일
                            dates = [base_date + timedelta(days=i) for i in range(4, 7)]  # 금~일
                        
                        # 중복 체크
                        for d in dates:
                            date_str = d.strftime('%Y-%m-%d')
                            if d in all_dates:
                                overlaps.append({
                                    'send_date': today.strftime('%Y-%m-%d'),
                                    'query_date': date_str,
                                    'previous_send': date_sources.get(d)
                                })
                            else:
                                all_dates.add(d)
                                date_sources[d] = today.strftime('%Y-%m-%d')
                except:
                    pass
    
    # 누락 체크 (연속성 확인)
    sorted_dates = sorted(all_dates)
    gaps = []
    for i in range(len(sorted_dates) - 1):
        diff = (sorted_dates[i+1] - sorted_dates[i]).days
        if diff > 1:
            gaps.append({
                'from': sorted_dates[i].strftime('%Y-%m-%d'),
                'to': sorted_dates[i+1].strftime('%Y-%m-%d'),
                'missing_days': diff - 1,
                'missing_dates': [(sorted_dates[i] + timedelta(days=j+1)).strftime('%Y-%m-%d') 
                                  for j in range(diff-1)]
            })
    
    return overlaps, gaps, sorted_dates

# 2026~2029년 검증 (4년)
overlaps, gaps, all_covered = check_coverage_v2(2026, 2040)

print(f"총 조회된 날짜 수: {len(all_covered)}")
print(f"중복 발생: {len(overlaps)}건")
if overlaps:
    print("중복 예시:")
    for overlap in overlaps[:3]:
        print(f"  발송일 {overlap['send_date']} -> 조회일 {overlap['query_date']} (이미 {overlap['previous_send']}에 조회됨)")

print(f"\n누락 발생: {len(gaps)}건")
if gaps:
    print("누락 예시:")
    for gap in gaps[:3]:
        print(f"  {gap['from']} 다음 {gap['to']} ({gap['missing_days']}일 누락)")
        print(f"    누락 날짜: {gap['missing_dates']}")

# 성공 메시지
if len(overlaps) == 0 and len(gaps) == 0:
    print("\n✅ 완벽합니다! 4년간 중복/누락 없음!")